# [8.1] Activation Patching Refresher - Exercises

Activation patching asks a concrete causal question: if the clean prompt has the behavior we want and the corrupt prompt does not, which clean activation slices recover the clean behavior when patched into the corrupt run?

The section-scale CUDA result is deliberately small and inspectable:

```text
clean:   The cat sat on the      target token: " floor"
corrupt: The bird flew over the  distractor:    " top"
patch scores by residual position: [0, 0, 0, 0, 0, 1]
```

A section-ready result is not just green tests. You should be able to explain why `0 = corrupt`, `1 = clean`, and why final-position-only recovery is a mechanics preflight rather than a complete circuit discovery.

<details>
<summary>Expected output</summary>

By the end of the notebook, your CPU smoke report should contain `logit_diff = 2.0`, recovered fraction `0.8`, sweep scores `[0.2, 0.8, 0.4]`, target localization passing, and top patches beating both average and max wrong-position controls. The committed CUDA report should show `[0, 0, 0, 0, 0, 1]` for residual-position patch scores.

</details>

<details>
<summary>Help - how this mirrors original ARENA activation patching</summary>

Original ARENA introduces clean/corrupt runs, a signed logit-diff metric, denoising patches, normalized recovery, and then increasingly fine component patching. This refresher extracts the reusable contract: metric -> patch -> recovered fraction -> sweep -> localization/control -> real CUDA report.

</details>


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch as t

chapter = "chapter8_automated_circuits"
section = "part1_activation_patching_refresher"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_activation_patching_refresher.tests as tests
import part1_activation_patching_refresher.utils as utils


## Logit-Diff Readouts

A patching experiment needs one scalar behavior metric. Here the metric is positive-minus-negative answer logit difference, averaged over any batch dimensions.

> ```yaml
> Difficulty: easy
> Importance: high
> You should spend 5 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_answer_logit_diff_validates_token_ids` passed!
All tests in `test_answer_logit_diff_rejects_degenerate_metrics` passed!
```

The toy logits `[[4, 1], [3, 2]]` should produce logit diff `2.0` for positive token `0` and negative token `1`.

</details>

<details>
<summary>Help - why logit diff?</summary>

The metric defines the task direction. If you swap the positive and negative tokens, every recovery score flips sign. Keep the answer pair fixed for clean, corrupt, and patched logits.

</details>

<details>
<summary>Common bug</summary>

Returning per-example differences instead of a scalar mean will pass some intuitive checks but break the recovery report, which expects one metric per run.

</details>

<details>
<summary>Solution sketch</summary>

Validate the vocabulary dimension and token ids, reject same-token and non-finite comparisons, then return `mean(logits[..., positive] - logits[..., negative])` as a Python float.

</details>


In [ ]:
def answer_logit_diff(
    logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
) -> float:
    raise NotImplementedError()


tests.test_answer_logit_diff_validates_token_ids(answer_logit_diff)
tests.test_answer_logit_diff_rejects_degenerate_metrics(answer_logit_diff)


## Clean-Into-Corrupt Patches

Denoising patches clean activations into the corrupt run and asks whether clean behavior returns. This helper is intentionally axis-generic: the component could be a sequence position today and a head, layer, or feature later.

> ```yaml
> Difficulty: easy
> Importance: high
> You should spend 5-10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_patch_activation_slice_replaces_one_component_without_mutating_inputs` passed!
```

The toy patch should produce `[[0.0, 0.0], [30.0, 40.0]]` while leaving the corrupt input unchanged.

</details>

<details>
<summary>Help - denoising vs noising</summary>

Denoising asks whether a clean activation is sufficient to restore clean behavior. Noising asks whether a corrupt activation is necessary by destroying clean behavior. This section uses denoising because later automated-circuit methods rank components by recovered behavior.

</details>

<details>
<summary>Common bug</summary>

In-place edits to the corrupt tensor contaminate future patches. Clone first, patch exactly one slice, and return the clone.

</details>

<details>
<summary>Solution sketch</summary>

Check matching shapes, validate `component_dim` and `component_index`, build a slice list, and copy the clean slice into a cloned corrupt tensor.

</details>


In [ ]:
def patch_activation_slice(
    clean_activations: t.Tensor,
    corrupt_activations: t.Tensor,
    *,
    component_index: int,
    component_dim: int = 0,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_patch_activation_slice_replaces_one_component_without_mutating_inputs(
    patch_activation_slice,
)


## Recovered Fractions And Sweeps

Normalize patched metrics by the signed clean-corrupt gap:

```text
recovered = (patched_metric - corrupt_metric) / (clean_metric - corrupt_metric)
```

For denoising, `0` means the corrupt behavior remained and `1` means the clean behavior was recovered.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10-15 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_patching_recovery_report_and_sweep_normalize_by_clean_corrupt_gap` passed!
All tests in `test_recovery_and_sweep_reject_degenerate_inputs` passed!
```

The toy clean/corrupt/patched metrics `3.0`, `-2.0`, and `2.0` should give recovered fraction `0.8`; patched metrics `[-1, 2, 0]` should give sweep scores `[0.2, 0.8, 0.4]` and best index `1`.

</details>

<details>
<summary>Help - I am confused about recovery scores</summary>

Recovery is a coordinate system, not a probability. The corrupt run sits at `0`; the clean run sits at `1`. Negative values mean the patch moved further in the corrupt direction, and values above `1` mean the patch overshot the clean metric.

</details>

<details>
<summary>Common bug</summary>

Do not divide by `abs(clean - corrupt)`. The metric sign carries the behavioral direction, and removing it can make harmful patches look helpful.

</details>

<details>
<summary>Solution sketch</summary>

Reject non-finite metrics and zero clean-corrupt gaps. For sweeps, require a nonempty rank-1 tensor and apply the same normalization to every patched metric.

</details>


In [ ]:
@dataclass(frozen=True)
class PatchingRecoveryReport:
    clean_metric: float
    corrupt_metric: float
    patched_metric: float
    recovered_fraction: float
    passes_recovery: bool


@dataclass(frozen=True)
class ActivationPatchingSweep:
    patch_scores: t.Tensor
    best_index: int
    best_score: float


def recovery_fraction(
    *,
    clean_metric: float,
    corrupt_metric: float,
    patched_metric: float,
) -> float:
    raise NotImplementedError()


def patching_recovery_report(
    clean_logits: t.Tensor,
    corrupt_logits: t.Tensor,
    patched_logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
    min_recovered_fraction: float = 0.5,
) -> PatchingRecoveryReport:
    raise NotImplementedError()


def activation_patching_sweep(
    *,
    clean_metric: float,
    corrupt_metric: float,
    patched_metrics: t.Tensor,
) -> ActivationPatchingSweep:
    raise NotImplementedError()


tests.test_patching_recovery_report_and_sweep_normalize_by_clean_corrupt_gap(
    patching_recovery_report,
    activation_patching_sweep,
    recovery_fraction,
)
tests.test_recovery_and_sweep_reject_degenerate_inputs(
    patching_recovery_report,
    activation_patching_sweep,
)


## Localization And Wrong-Position Controls

Patching evidence needs controls. In the real CUDA run, the target is final residual position `5`; the wrong-position control set is positions `0..4`. The final patch must beat both the mean wrong-position score and the largest individual wrong-position score.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10-15 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_localization_and_random_controls_require_top_components_to_win` passed!
All tests in `test_localization_and_random_controls_reject_bad_indices` passed!
```

For toy scores `[0.2, 0.9, 0.8, 0.1]`, top indices should be `(1, 2)`, the random-control mean should be `0.15`, and the max random-control score should be `0.2`.

</details>

<details>
<summary>Help - why track max control?</summary>

Averaging controls can hide a single wrong component with a high patch score. The max-control check makes the claim sharper: the best target patch must beat every wrong-position patch, not merely their average.

</details>

<details>
<summary>Common bug</summary>

Treating equality as success weakens the control. A target patch tied with a wrong-position patch has not localized the behavior.

</details>

<details>
<summary>Solution sketch</summary>

Use `topk` to get top indices, compute overlap against validated targets, then compare top patch score to both the mean and max over validated wrong-control indices.

</details>


In [ ]:
@dataclass(frozen=True)
class PatchingLocalizationReport:
    top_indices: tuple[int, ...]
    target_indices: tuple[int, ...]
    topk_overlap: float
    localizes_target: bool


@dataclass(frozen=True)
class RandomPatchControlReport:
    top_patch_score: float
    random_patch_score: float
    max_random_patch_score: float
    top_beats_random: bool
    top_beats_max_random: bool


def patching_localization_report(
    patch_scores: t.Tensor,
    target_indices: list[int],
    *,
    top_k: int = 2,
    min_overlap: float = 0.5,
) -> PatchingLocalizationReport:
    raise NotImplementedError()


def random_patch_control_report(
    patch_scores: t.Tensor,
    random_indices: list[int],
    *,
    top_k: int = 2,
) -> RandomPatchControlReport:
    raise NotImplementedError()


tests.test_localization_and_random_controls_require_top_components_to_win(
    patching_localization_report,
    random_patch_control_report,
)
tests.test_localization_and_random_controls_reject_bad_indices(
    patching_localization_report,
    random_patch_control_report,
)


## Combined Contract

Compose the local helpers into a CPU-only smoke report before touching the live model. The shape should mirror the CUDA evidence: metric, patch, recovery, sweep, localization, and wrong-position controls.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

The contract should expose `logit_diff = 2.0`, recovered fraction `0.8`, sweep scores `[0.2, 0.8, 0.4]`, `localizes_target = True`, `top_beats_random = True`, and `top_beats_max_random = True`.

</details>

<details>
<summary>Help - reading the combined contract</summary>

The smoke report is not model evidence. It is a debugging contract that proves your implementation has the same objects the model path will need.

</details>

<details>
<summary>Solution sketch</summary>

Use the functions above with the toy logits and patch scores from the tests, convert dataclasses to dictionaries, and return a JSON-like report.

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)


## Signature Result

The committed CUDA report is the section-scale result. It should show a pinned `gelu-1l` TransformerLens run where final residual-position patching exactly recovers the clean final-token logits and every non-final wrong-position patch stays at zero.

<details>
<summary>Expected output</summary>

```text
preflight_passed: true
patch_scores_by_position: [0, 0, 0, 0, 0, 1]
target_recovered_fraction: 1.0
wrong_position_control_fraction: 0.0
max_wrong_position_control_fraction: 0.0
top_beats_max_wrong_position_control: true
```

</details>

<details>
<summary>Question - what does final-position-only recovery mean?</summary>

It means the final residual vector already contains the next-token information needed by the unembedding for this prompt pair. It does not identify the upstream heads, MLPs, features, or edges that produced that information.

</details>

<details>
<summary>Common bug</summary>

Do not call this IOI-scale circuit discovery. This is a residual-stream mechanics preflight.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["preflight_passed"]
    assert gpu["patch_scores_by_position"] == [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
    assert gpu["best_position"] == gpu["target_position"] == 5
    assert gpu["target_recovered_fraction"] >= 0.99
    assert abs(gpu["wrong_position_control_fraction"]) <= 1e-4
    assert abs(gpu["max_wrong_position_control_fraction"]) <= 1e-4
    assert gpu["top_beats_wrong_position_control"]
    assert gpu["top_beats_max_wrong_position_control"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test(max_vram_gb=24.0)
utils.print_report(
    "Committed CUDA activation-patching report",
    {
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "device": gpu["device"],
        "patch_scores": gpu["patch_scores_by_position"],
        "target_recovered_fraction": gpu["target_recovered_fraction"],
        "max_wrong_position_control": gpu["max_wrong_position_control_fraction"],
        "peak_vram_gb": round(gpu["peak_vram_gb"], 4),
    },
)


## Limitations

This is a GT-1 activation-patching mechanics preflight on one pinned `gelu-1l` hook and one safe prompt pair. It is not a full IOI circuit analysis, not an OOD prompt-template benchmark, and not a feature-level attribution graph.

## Further Research

Repeat the sweep across prompt templates, compare denoising and noising patches, patch blocks/heads/QKV/patterns, and use this contract as the baseline for attribution patching, ACDC, EAP, and sparse feature circuits in the next Chapter 8 sections.
